## Step 3
Input: s3://thesis--ec331-s3/FCAS_RAISE1SEC-Price-Bids/
Output: s3://thesis--ec331-s3/melted-volume-bids/

In [ ]:
import awswrangler as wr
import pandas as pd

# Path to the price bids file
price_bids_path = "s3://thesis--ec331-s3/FCAS_RAISE1SEC-Price-Bids/RAISE1SEC_FILTERED_202310010000.parquet"

# Read the price bids file
try:
    price_bids_df = wr.s3.read_parquet(path=price_bids_path)
    print(f"Successfully read price bids file: {price_bids_path}")
    print(f"Shape: {price_bids_df.shape}")
    print(price_bids_df.head())
except Exception as e:
    print(f"Error reading price bids file: {e}")
    
    # If the above fails, check if it's a folder of Parquet files
    try:
        price_bids_folder_path = price_bids_path + "/"
        price_bids_df = wr.s3.read_parquet(path=price_bids_folder_path)
        print(f"Successfully read price bids folder: {price_bids_folder_path}")
        print(f"Shape: {price_bids_df.shape}")
        print(price_bids_df.head())
    except Exception as e2:
        print(f"Error reading as folder: {e2}")

In [ ]:
price_bids_df.columns

In [ ]:
# To find the first (earliest) datetime
first_datetime = price_bids_df['SETTLEMENTDATE'].min()

# To find the last (latest) datetime
last_datetime = price_bids_df['SETTLEMENTDATE'].max()

# Print the results
print(f"First datetime: {first_datetime}")
print(f"Last datetime: {last_datetime}")

In [ ]:
import pandas as pd
import awswrangler as wr
import time
from datetime import datetime, timedelta
import gc
import os
import psutil
import boto3
import glob

def get_memory_usage():
    """Return the current memory usage of the process in GB"""
    process = psutil.Process(os.getpid())
    memory_gb = process.memory_info().rss / 1024 / 1024 / 1024
    return memory_gb

print(f"Initial memory usage: {get_memory_usage():.2f} GB")

# Define which columns to melt
price_band_cols = [f"PRICEBAND{i}" for i in range(1, 11)]

def melt_and_write_chunks(df, chunk_size=50000, timestamp=None):
    """
    Melts the PRICEBAND columns in chunks and writes each chunk directly to S3
    """
    if timestamp is None:
        timestamp = datetime.now().strftime("%Y%m%d%H%M%S")
        
    print(f"Starting streaming melt of dataframe with shape: {df.shape}")
    print(f"Current memory usage: {get_memory_usage():.2f} GB")
    start_time = time.time()
    
    # Calculate number of chunks
    num_chunks = (len(df) + chunk_size - 1) // chunk_size
    print(f"Processing in {num_chunks} chunks of size {chunk_size}")
    
    # Create output path
    output_base = f"s3://your-bucket-name/melted-price-bids/enriched_price_bids_melted_{timestamp}"
    
    # Track total rows processed
    total_rows_processed = 0
    all_chunk_files = []
    
    # Process dataframe in chunks
    for i in range(num_chunks):
        chunk_start = i * chunk_size
        chunk_end = min((i + 1) * chunk_size, len(df))
        
        print(f"Processing chunk {i+1}/{num_chunks} (rows {chunk_start} to {chunk_end-1})")
        print(f"Memory before chunk processing: {get_memory_usage():.2f} GB")
        
        # Extract chunk and immediately release the original slice reference
        chunk = df.iloc[chunk_start:chunk_end].copy()
        
        # Identify columns not being melted
        id_vars_cols = [col for col in chunk.columns if col not in price_band_cols]
        
        # Keep only necessary columns
        chunk = chunk[id_vars_cols + [col for col in price_band_cols if col in chunk.columns]]
        
        # Melt this chunk
        chunk_start_time = time.time()
        chunk_melted = pd.melt(
            chunk,
            id_vars=id_vars_cols,
            value_vars=[col for col in price_band_cols if col in chunk.columns],
            var_name="BIDBAND",
            value_name="BIDPRICE"
        )
        
        # Clean up original chunk to free memory
        del chunk
        gc.collect()
        
        # Extract the band number
        chunk_melted["BIDBAND"] = chunk_melted["BIDBAND"].str.extract(r"(\d+)").astype(int)
        
        # Apply any necessary transformations from the original script
        if "BIDTYPE" in chunk_melted.columns:
            chunk_melted["BIDTYPE"] = chunk_melted["BIDTYPE"].astype(str)
            
        if "DUID" in chunk_melted.columns:
            chunk_melted["DUID"] = chunk_melted["DUID"].astype(str)
            
        if "SETTLEMENTDATE" in chunk_melted.columns:
            chunk_melted["SETTLEMENTDATE"] = pd.to_datetime(chunk_melted["SETTLEMENTDATE"])
            # Add 4 hours, 5 minutes offset for APPLICABLEFROM
            chunk_melted["APPLICABLEFROM"] = chunk_melted["SETTLEMENTDATE"] + timedelta(hours=4, minutes=5)
            chunk_melted["APPLICABLEFROM"] = pd.to_datetime(chunk_melted["APPLICABLEFROM"])
        
        # Filter out null values to reduce size
        chunk_melted = chunk_melted.dropna(subset=["BIDPRICE"])
        
        # Count rows in this chunk
        chunk_rows = len(chunk_melted)
        total_rows_processed += chunk_rows
        
        # Write this chunk directly to S3
        chunk_output = f"{output_base}_part{i+1:04d}.parquet"
        write_start = time.time()
        
        try:
            wr.s3.to_parquet(
                df=chunk_melted,
                path=chunk_output,
                index=False,
                compression="snappy"
            )
            all_chunk_files.append(chunk_output)
            write_time = time.time() - write_start
            print(f"  ✓ Chunk {i+1} written to S3 in {write_time:.2f} seconds ({chunk_rows} rows)")
        except Exception as e:
            print(f"  ✗ Error writing chunk {i+1} to S3: {str(e)}")
        
        # Clean up melted chunk to free memory
        del chunk_melted
        gc.collect()
        
        print(f"  Memory after chunk processing: {get_memory_usage():.2f} GB")
        print(f"  Chunk {i+1} processed in {time.time() - chunk_start_time:.2f} seconds")
    
    total_time = time.time() - start_time
    print(f"All chunks processed and written in {total_time:.2f} seconds")
    print(f"Total rows processed: {total_rows_processed}")
    print(f"Final memory usage: {get_memory_usage():.2f} GB")
    
    return output_base, all_chunk_files, total_rows_processed

def process_local_files():
    """Process local Parquet files and upload melted results to S3"""
    # Create timestamp for consistent file naming
    timestamp = datetime.now().strftime("%Y%m%d%H%M%S")
    
    # Define your input directory where price Parquet files live
    input_dir = "/Volumes/T7/bid-price-data-excluding-energy-B2"
    
    # Gather Parquet files
    parquet_files = sorted(glob.glob(os.path.join(input_dir, "*.parquet")))
    
    # Exclude hidden macOS files
    parquet_files = [f for f in parquet_files if not os.path.basename(f).startswith("._")]
    print(f"Found {len(parquet_files)} Parquet files in {input_dir}.")
    
    # Process the files in batches to avoid memory issues
    batch_size = 10  # Number of files to process at once
    num_batches = (len(parquet_files) + batch_size - 1) // batch_size
    
    print(f"Will process files in {num_batches} batches of up to {batch_size} files each")
    
    # Create a list to store paths of all melted file manifests and all melted files
    melted_manifests = []
    all_melted_files = []
    global_total_rows = 0  # To accumulate total rows processed across batches
    
    # Process each batch
    for batch_num in range(num_batches):
        batch_start = batch_num * batch_size
        batch_end = min((batch_num + 1) * batch_size, len(parquet_files))
        batch_files = parquet_files[batch_start:batch_end]
        
        print(f"\nProcessing batch {batch_num + 1}/{num_batches} with {len(batch_files)} files")
        print(f"Memory before batch processing: {get_memory_usage():.2f} GB")
        
        try:
            # Read this batch of files
            print(f"Reading batch of {len(batch_files)} files...")
            batch_start_time = time.time()
            
            # Create an empty list to hold DataFrames
            batch_dfs = []
            
            # Read each file in the batch
            for file_path in batch_files:
                try:
                    df = pd.read_parquet(file_path)
                    batch_dfs.append(df)
                    print(f"  Read {file_path} with shape {df.shape}")
                except Exception as e:
                    print(f"  Error reading {file_path}: {str(e)}")
            
            # Concatenate all DataFrames in the batch
            if batch_dfs:
                batch_df = pd.concat(batch_dfs, ignore_index=True)
                print(f"Read batch in {time.time() - batch_start_time:.2f} seconds")
                print(f"Batch data shape: {batch_df.shape}")
                print(f"Memory after reading batch: {get_memory_usage():.2f} GB")
                
                # Clean up the list of DataFrames to free memory
                del batch_dfs
                gc.collect()
                
                # Melt and write this batch
                batch_timestamp = f"{timestamp}_batch{batch_num+1:03d}"
                output_base, batch_melted_files, batch_rows = melt_and_write_chunks(batch_df, chunk_size=50000, timestamp=batch_timestamp)
                
                # Add these files to our master list and update the row count
                all_melted_files.extend(batch_melted_files)
                global_total_rows += batch_rows
                
                # Clean up to free memory
                del batch_df
                gc.collect()
                
                # Create a manifest file for this batch
                try:
                    manifest = pd.DataFrame({"file_path": batch_melted_files})
                    manifest_path = f"{output_base}_manifest.csv"
                    wr.s3.to_csv(manifest, manifest_path, index=False)
                    print(f"Created manifest file for batch {batch_num + 1}: {manifest_path}")
                    
                    # Add this manifest to our list
                    melted_manifests.append(manifest_path)
                except Exception as e:
                    print(f"Error creating manifest file for batch {batch_num + 1}: {str(e)}")
            else:
                print(f"No data to process in batch {batch_num + 1}")
            
        except Exception as e:
            print(f"Error processing batch {batch_num + 1}: {str(e)}")
        
        print(f"Completed batch {batch_num + 1}/{num_batches}")
        print(f"Memory after batch processing: {get_memory_usage():.2f} GB")
    
    print("\nAll batches processed!")
    print(f"Total melted files created: {len(all_melted_files)}")
    
    # Create a master manifest of all manifests
    try:
        if melted_manifests:
            master_manifest = pd.DataFrame({"manifest_path": melted_manifests})
            master_manifest_path = f"s3://your-bucket-name/melted-price-bids/master_manifest_{timestamp}.csv"
            wr.s3.to_csv(master_manifest, master_manifest_path, index=False)
            print(f"Created master manifest at: {master_manifest_path}")
            return master_manifest_path, all_melted_files, global_total_rows
    except Exception as e:
        print(f"Error creating master manifest: {str(e)}")
    
    return None, all_melted_files, global_total_rows

def process_s3_files():
    """Process Parquet files directly from S3"""
    # Create timestamp for consistent file naming
    timestamp = datetime.now().strftime("%Y%m%d%H%M%S")
    
    # Locate all the price bids files in S3 using boto3 for better pagination
    print("Identifying price bids files in S3...")
    try:
        s3_client = boto3.client('s3')
        bucket = "your-bucket-name"
        prefix = "price-bids/"  # Adjust this to where your price bids are stored
        price_files = []
        
        # Use pagination to handle large number of files
        paginator = s3_client.get_paginator('list_objects_v2')
        page_iterator = paginator.paginate(Bucket=bucket, Prefix=prefix)
        
        for page in page_iterator:
            if 'Contents' in page:
                for obj in page['Contents']:
                    if obj['Key'].endswith('.parquet'):
                        file_path = f"s3://{bucket}/{obj['Key']}"
                        price_files.append(file_path)
        
        print(f"Found {len(price_files)} price bids files")
        
        if not price_files:
            raise ValueError("No price bids files found in S3")
            
    except Exception as e:
        print(f"Error listing price bids files: {str(e)}")
        raise
    
    # Process the files in batches to avoid memory issues
    batch_size = 10  # Number of files to process at once
    num_batches = (len(price_files) + batch_size - 1) // batch_size
    
    print(f"Will process files in {num_batches} batches of up to {batch_size} files each")
    
    # Create a list to store paths of all melted file manifests and all melted files
    melted_manifests = []
    all_melted_files = []
    global_total_rows = 0  # To accumulate total rows processed across batches
    
    # Process each batch
    for batch_num in range(num_batches):
        batch_start = batch_num * batch_size
        batch_end = min((batch_num + 1) * batch_size, len(price_files))
        batch_files = price_files[batch_start:batch_end]
        
        print(f"\nProcessing batch {batch_num + 1}/{num_batches} with {len(batch_files)} files")
        print(f"Memory before batch processing: {get_memory_usage():.2f} GB")
        
        try:
            # Read this batch of files
            print(f"Reading batch of {len(batch_files)} files...")
            batch_start_time = time.time()
            
            # Read the batch of parquet files using dataset=False because batch_files is a list
            batch_df = wr.s3.read_parquet(
                path=batch_files,
                dataset=False
            )
            
            print(f"Read batch in {time.time() - batch_start_time:.2f} seconds")
            print(f"Batch data shape: {batch_df.shape}")
            print(f"Memory after reading batch: {get_memory_usage():.2f} GB")
            
            # Melt and write this batch
            batch_timestamp = f"{timestamp}_batch{batch_num+1:03d}"
            output_base, batch_melted_files, batch_rows = melt_and_write_chunks(batch_df, chunk_size=50000, timestamp=batch_timestamp)
            
            # Add these files to our master list and update the row count
            all_melted_files.extend(batch_melted_files)
            global_total_rows += batch_rows
            
            # Clean up to free memory
            del batch_df
            gc.collect()
            
            # Create a manifest file for this batch
            try:
                manifest = pd.DataFrame({"file_path": batch_melted_files})
                manifest_path = f"{output_base}_manifest.csv"
                wr.s3.to_csv(manifest, manifest_path, index=False)
                print(f"Created manifest file for batch {batch_num + 1}: {manifest_path}")
                
                # Add this manifest to our list
                melted_manifests.append(manifest_path)
            except Exception as e:
                print(f"Error creating manifest file for batch {batch_num + 1}: {str(e)}")
            
        except Exception as e:
            print(f"Error processing batch {batch_num + 1}: {str(e)}")
        
        print(f"Completed batch {batch_num + 1}/{num_batches}")
        print(f"Memory after batch processing: {get_memory_usage():.2f} GB")
    
    print("\nAll batches processed!")
    print(f"Total melted files created: {len(all_melted_files)}")
    
    # Create a master manifest of all manifests
    try:
        if melted_manifests:
            master_manifest = pd.DataFrame({"manifest_path": melted_manifests})
            master_manifest_path = f"s3://your-bucket-name/melted-price-bids/master_manifest_{timestamp}.csv"
            wr.s3.to_csv(master_manifest, master_manifest_path, index=False)
            print(f"Created master manifest at: {master_manifest_path}")
            return master_manifest_path, all_melted_files, global_total_rows
    except Exception as e:
        print(f"Error creating master manifest: {str(e)}")
    
    return None, all_melted_files, global_total_rows

def main():
    # Choose which data source to use: 'local' or 's3'
    data_source = 'local'  # Change to 's3' if your files are already in S3
    
    if data_source == 'local':
        master_manifest_path, all_melted_files, global_total_rows = process_local_files()
    else:
        master_manifest_path, all_melted_files, global_total_rows = process_s3_files()
    
    # Now, try to create a final DataFrame with all the melted data
    print("\nLoading all melted data into final_df...")
    try:
        # First, check if it's feasible to load all data at once
        est_rows_per_file = global_total_rows / len(all_melted_files) if all_melted_files else 0
        est_total_rows = est_rows_per_file * len(all_melted_files)
        
        print(f"Estimated total rows in all melted files: {est_total_rows:,.0f}")
        
        # Alternative: Load a sample first to estimate memory requirements
        sample_size = min(5, len(all_melted_files))
        if sample_size > 0:
            print(f"Loading sample of {sample_size} files to estimate memory requirements...")
            sample_df = wr.s3.read_parquet(path=all_melted_files[:sample_size], dataset=False)
            bytes_per_row = sample_df.memory_usage(deep=True).sum() / len(sample_df)
            est_memory_gb = (bytes_per_row * est_total_rows) / 1e9
            
            print(f"Sample loaded: {len(sample_df)} rows")
            print(f"Estimated memory required for full dataset: {est_memory_gb:.2f} GB")
            
            # Show sample data
            print("\nSample data preview:")
            print(sample_df.head())
            
            # Clean up sample
            del sample_df
            gc.collect()
        
        # Load all the data if it seems feasible or force loading
        force_load = False  # Set to True to load everything even if it might be large
        
        if force_load or (est_memory_gb < get_memory_usage() * 5):  # If estimated size is reasonable
            print("\nLoading all melted data...")
            final_df = wr.s3.read_parquet(path=all_melted_files, dataset=False)
            print(f"Successfully loaded all data into final_df")
            print(f"Final dataframe shape: {final_df.shape}")
            print(f"Final dataframe columns: {final_df.columns.tolist()}")
            print(f"Final dataframe preview:\n{final_df.head()}")
            
            # Optional: Save the final DataFrame to a single file for easier access later
            final_output_path = f"s3://your-bucket-name/melted-price-bids/combined_melted_data_{datetime.now().strftime('%Y%m%d%H%M%S')}.parquet"
            print(f"\nSaving final_df to {final_output_path}...")
            wr.s3.to_parquet(
                df=final_df,
                path=final_output_path,
                index=False,
                compression="snappy"
            )
            print(f"Successfully saved final_df to {final_output_path}")
        else:
            print(f"\nWarning: Full dataset would require approximately {est_memory_gb:.2f} GB of memory.")
            print("To load all data, set force_load = True in the code.")
            print(f"You can still access all the individual melted files from {master_manifest_path}")
            
    except Exception as e:
        print(f"Error loading melted data: {str(e)}")
        print("The melted data is still available in the individual files.")
        print(f"You can find all file paths in the master manifest: {master_manifest_path}")

    print("\nProcess complete!")
    print(f"Final memory usage: {get_memory_usage():.2f} GB")

if __name__ == "__main__":
    main()